# 05 — Cython, Numba et mypyc : teasers

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :
- expliquer pourquoi Python pur est lent pour le calcul numérique intensif ;
- utiliser **Numba** (`@jit`) pour accélérer une fonction numérique sans modifier le code ;
- comprendre le principe de **Cython** (compilation AOT en C) ;
- découvrir **mypyc** (compilation AOT depuis les type hints) ;
- choisir l'outil adapté à votre cas d'usage.

## Prérequis — ce que vous connaissez déjà

Ce notebook s'adresse à un développeur Python **confirmé**. Vous maîtrisez déjà :
- le profiling CPU et mémoire (notebooks 01-02) ;
- le benchmarking (notebook 03) ;
- les techniques d'optimisation Python pur (notebook 04) ;
- les type hints et les annotations.

## Plan

1. Pourquoi Python est lent en calcul intensif ?
2. Numba — JIT compilation pour le numérique
3. Cython — compilation AOT en C
4. mypyc — compilation depuis les type hints
5. Comparaison et choix d'outil
6. Quand rester en Python pur
7. Synthèse
8. Exercices
9. Ressources

---

## 1. Pourquoi Python est lent en calcul intensif ?

Python est un langage **interprété** avec des types **dynamiques**. Chaque opération arithmétique implique :

1. Vérification du type à runtime ;
2. Dispatch vers la bonne implémentation (`__add__`, `__mul__`, etc.) ;
3. Allocation d'un nouvel objet Python pour le résultat ;
4. Mise à jour du compteur de références.

Pour une addition `a + b` entre deux `int`, C fait **1 instruction CPU**, Python fait **~100 instructions**.

In [ ]:
import timeit

# Somme en Python pur
def somme_python(n):
    total = 0
    for i in range(n):
        total += i
    return total

t = timeit.timeit(lambda: somme_python(10_000_000), number=1)
print(f"Python pur : {t:.3f}s pour 10M additions")

L'objectif des outils présentés ici est de **compiler** les boucles critiques en code machine natif, tout en gardant la syntaxe Python.

---

## 2. Numba — JIT compilation pour le numérique

**Numba** compile des fonctions Python en code machine **à la volée** (JIT — Just In Time) grâce à LLVM. Il supporte un sous-ensemble de Python (principalement NumPy et les boucles numériques).

> **Installation :** `pip install numba`

### Premier exemple : `@jit`

In [ ]:
try:
    from numba import jit

    @jit(nopython=True)
    def somme_numba(n):
        total = 0
        for i in range(n):
            total += i
        return total

    # Premier appel : compilation (lent)
    somme_numba(10)

    # Appels suivants : code machine natif
    t = timeit.timeit(lambda: somme_numba(10_000_000), number=1)
    print(f"Numba : {t:.4f}s pour 10M additions")

    t_py = timeit.timeit(lambda: somme_python(10_000_000), number=1)
    print(f"Python pur : {t_py:.3f}s")
    print(f"Accélération : {t_py / t:.0f}x")

except ImportError:
    print("numba non installé — pip install numba")

### `nopython=True` vs `object` mode

| Mode | Signification | Performance |
|---|---|---|
| `nopython=True` | Tout doit être compilable en machine | Maximale |
| `object` mode (fallback) | Utilise l'API C de Python | Quasi nulle |

**Toujours utiliser `nopython=True`** (ou le raccourci `@njit`). Si Numba ne peut pas compiler, vous voulez une erreur, pas un fallback silencieux.

### `@njit` — raccourci

In [ ]:
try:
    from numba import njit

    @njit
    def produit_scalaire(a, b):
        total = 0.0
        for i in range(len(a)):
            total += a[i] * b[i]
        return total

    import numpy as np
    a = np.random.rand(1_000_000)
    b = np.random.rand(1_000_000)

    # Warm up
    produit_scalaire(a[:10], b[:10])

    t_numba = timeit.timeit(lambda: produit_scalaire(a, b), number=100)
    t_numpy = timeit.timeit(lambda: np.dot(a, b), number=100)

    print(f"Numba  : {t_numba:.3f}s")
    print(f"NumPy  : {t_numpy:.3f}s")
    print(f"Ratio  : {t_numba / t_numpy:.1f}x")

except ImportError:
    print("numba ou numpy non installé")

### Parallélisme automatique avec `parallel=True`

In [ ]:
try:
    from numba import njit, prange

    @njit(parallel=True)
    def somme_parallele(n):
        total = 0
        for i in prange(n):  # prange = parallel range
            total += i
        return total

    somme_parallele(10)  # warm-up
    t = timeit.timeit(lambda: somme_parallele(10_000_000), number=10)
    print(f"Numba parallèle : {t / 10 * 1000:.1f} ms")

except ImportError:
    print("numba non installé")

### Limitations de Numba

Numba **ne supporte pas** :
- les classes Python arbitraires (seulement `@jitclass` limité) ;
- les chaînes de caractères (support partiel) ;
- les dictionnaires Python (seulement `numba.typed.Dict`) ;
- les bibliothèques tierces (seul NumPy est supporté) ;
- les exceptions complexes.

**Numba est fait pour le calcul numérique en boucle.** Pas pour du code Python généraliste.

---

## 3. Cython — compilation AOT en C

**Cython** est un superset de Python qui compile en C. Contrairement à Numba (JIT), Cython compile **avant l'exécution** (AOT — Ahead Of Time).

> **Installation :** `pip install cython`

### Principe

```python
# fichier somme.pyx (syntaxe Cython)
def somme_cython(int n):
    cdef long total = 0
    cdef int i
    for i in range(n):
        total += i
    return total
```

Compilation :
```bash
cythonize -i somme.pyx
```

Le fichier `.pyx` est compilé en `.c` puis en `.so` (extension Python). La fonction est appelable normalement depuis Python.

### Annotations de type Cython

| Déclaration | Signification |
|---|---|
| `cdef int x` | Variable C locale (pas visible depuis Python) |
| `cdef double y` | Variable C flottante |
| `def f(int n)` | Paramètre typé (visible depuis Python) |
| `cpdef f(int n)` | Callable depuis Python et C |
| `cdef inline` | Fonction inlinée en C |

Les types C éliminent le dispatch dynamique et les allocations d'objets Python.

### Exemple en notebook avec `%%cython` magic

In [ ]:
# Pour utiliser %%cython en notebook :
# pip install cython
# %load_ext Cython

# Puis dans une cellule :
# %%cython
# def somme_cython(int n):
#     cdef long total = 0
#     cdef int i
#     for i in range(n):
#         total += i
#     return total

print("Cython nécessite %load_ext Cython + cellule %%cython")
print("Voir la documentation Cython pour l'utilisation en notebook")

### Quand choisir Cython ?

| Critère | Numba | Cython |
|---|---|---|
| Compilation | JIT (runtime) | AOT (build time) |
| Langages supportés | Python + NumPy | Python + C/C++ |
| Intégration C existant | Non | Oui (wrapping C/C++) |
| Distribution | Simple (pip) | Nécessite un compilateur C |
| Courbe d'apprentissage | Faible | Moyenne à élevée |
| Production | Oui | Oui (utilisé par pandas, scikit-learn) |

---

## 4. mypyc — compilation depuis les type hints

**mypyc** compile du Python **standard** annoté avec des type hints en extensions C. C'est le compilateur derrière `mypy` lui-même.

> **Installation :** `pip install mypy` (mypyc est inclus)

### Exemple

```python
# fichier calcul.py — du Python standard avec type hints
def somme_mypyc(n: int) -> int:
    total: int = 0
    for i in range(n):
        total += i
    return total
```

Compilation :
```bash
mypyc calcul.py
```

Le résultat est une extension `.so` importable normalement.

### Avantages de mypyc

- **Pas de syntaxe spéciale** : c'est du Python standard avec des type hints ;
- **Compatibilité mypy** : si votre code passe `mypy --strict`, il peut être compilé ;
- **Classes supportées** : contrairement à Numba, mypyc compile les classes ;
- **Utilisé en production** : mypy lui-même est compilé avec mypyc.

### Limitations de mypyc

- Pas de compilation JIT (AOT uniquement) ;
- Ne supporte pas `*args`/`**kwargs` dynamiques ;
- Certaines fonctionnalités dynamiques de Python sont interdites ;
- Moins mature que Cython pour le calcul numérique.

---

## 5. Comparaison et choix d'outil

| Critère | Python pur | Numba | Cython | mypyc |
|---|---|---|---|---|
| **Syntaxe** | Standard | Standard (subset) | Superset | Standard + hints |
| **Compilation** | Non | JIT | AOT | AOT |
| **Cas d'usage** | Tout | Numérique | Numérique + C | Généraliste |
| **Gain typique** | 1x | 10-200x | 10-200x | 2-5x |
| **Classes** | Oui | Limité | Oui | Oui |
| **Distribution** | Simple | Simple | Compilateur C requis | Simple |
| **Maturité** | — | Bonne | Excellente | En croissance |

### Arbre de décision

```
Le code est-il CPU-bound en boucle numérique ?
├── Oui → Numba (@njit) en premier
│         Si besoin de wrapper du C → Cython
├── Non, mais CPU-bound en logique Python
│   └── mypyc (si code typé) ou algorithme + structures
└── Non, I/O-bound
    └── asyncio / multiprocessing (pas de compilation)
```

---

## 6. Quand rester en Python pur

La compilation n'est **pas toujours la réponse**. Avant de sortir Numba ou Cython :

1. **Avez-vous profilé ?** Peut-être que le goulot n'est pas où vous pensez.
2. **Existe-t-il une bibliothèque optimisée ?** NumPy, pandas, polars, etc. sont déjà compilés.
3. **Le code est-il I/O-bound ?** La compilation n'aide pas si vous attendez le réseau ou le disque.
4. **Le code est-il assez critique ?** Si la fonction prend 1 ms, gagner 100x ne se voit pas.

In [ ]:
# NumPy est souvent la meilleure option pour le numérique
try:
    import numpy as np

    # Python pur
    def somme_carres_python(n):
        return sum(i * i for i in range(n))

    # NumPy (déjà compilé en C)
    def somme_carres_numpy(n):
        return np.sum(np.arange(n, dtype=np.int64) ** 2)

    n = 1_000_000
    t_py = timeit.timeit(lambda: somme_carres_python(n), number=10)
    t_np = timeit.timeit(lambda: somme_carres_numpy(n), number=10)
    print(f"Python pur : {t_py:.3f}s")
    print(f"NumPy      : {t_np:.3f}s")
    print(f"NumPy est {t_py / t_np:.0f}x plus rapide")

except ImportError:
    print("numpy non installé")

### Le coût de la complexité

| Solution | Performance | Maintenabilité | Débogage | Distribution |
|---|---|---|---|---|
| Python pur optimisé | Baseline | Excellente | Simple | `pip install` |
| NumPy/pandas | 10-100x | Bonne | Simple | `pip install` |
| Numba `@njit` | 10-200x | Bonne | Moyenne | `pip install` |
| Cython | 10-200x | Moyenne | Difficile | Compilateur C |
| Extension C/Rust | 100-500x | Faible | Difficile | Build system |

**N'ajoutez de la complexité que si le gain justifie le coût.**

---

## 7. Synthèse

| Outil | Type | Cas d'usage principal | Installation |
|---|---|---|---|
| **Numba** | JIT | Boucles numériques, NumPy | `pip install numba` |
| **Cython** | AOT | Calcul + wrapping C/C++ | `pip install cython` + compilateur |
| **mypyc** | AOT | Code typé généraliste | `pip install mypy` |
| **NumPy** | Bibliothèque | Opérations vectorielles | `pip install numpy` |

**Règles à retenir :**
- Profilez et optimisez l'algorithme **avant** de compiler.
- Numba est le plus simple pour le numérique (1 décorateur).
- Cython est le plus puissant pour l'intégration C.
- mypyc est prometteur pour le code généraliste typé.
- NumPy est souvent suffisant et plus simple que tout compilateur.
- La compilation n'aide **pas** pour le code I/O-bound.

---

## 8. Exercices

### Exercice 1 — Benchmarker Python pur vs NumPy *(facile)*

Comparez 3 façons de calculer la norme euclidienne d'un vecteur de 1 million d'éléments :
1. Boucle Python : `sqrt(sum(x_i ** 2))`
2. Compréhension Python : `sqrt(sum(x**2 for x in vec))`
3. NumPy : `np.linalg.norm(vec)`

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Cython_numba_teasers", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit
import math

try:
    import numpy as np

    vec_list = [float(i) for i in range(100_000)]
    vec_np = np.array(vec_list)

    def norme_boucle(v):
        total = 0.0
        for x in v:
            total += x * x
        return math.sqrt(total)

    def norme_comp(v):
        return math.sqrt(sum(x * x for x in v))

    def norme_numpy(v):
        return np.linalg.norm(v)

    t1 = timeit.timeit(lambda: norme_boucle(vec_list), number=10)
    t2 = timeit.timeit(lambda: norme_comp(vec_list), number=10)
    t3 = timeit.timeit(lambda: norme_numpy(vec_np), number=10)

    print(f"Boucle       : {t1:.3f}s")
    print(f"Compréhension: {t2:.3f}s")
    print(f"NumPy        : {t3:.4f}s")
    print(f"NumPy est {t1/t3:.0f}x plus rapide que la boucle")

except ImportError:
    print("numpy non installé")
```

</details>

### Exercice 2 — Accélérer avec Numba *(moyen)*

Écrire une fonction `mandelbrot_pixel(c_re, c_im, max_iter)` qui calcule le nombre d'itérations pour un point du plan complexe. Comparez la version Python pure et la version Numba.

```python
def mandelbrot_pixel(c_re, c_im, max_iter):
    z_re, z_im = 0.0, 0.0
    for i in range(max_iter):
        z_re2 = z_re * z_re
        z_im2 = z_im * z_im
        if z_re2 + z_im2 > 4.0:
            return i
        z_im = 2 * z_re * z_im + c_im
        z_re = z_re2 - z_im2 + c_re
    return max_iter
```

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Cython_numba_teasers", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import timeit

def mandelbrot_python(c_re, c_im, max_iter):
    z_re, z_im = 0.0, 0.0
    for i in range(max_iter):
        z_re2 = z_re * z_re
        z_im2 = z_im * z_im
        if z_re2 + z_im2 > 4.0:
            return i
        z_im = 2 * z_re * z_im + c_im
        z_re = z_re2 - z_im2 + c_re
    return max_iter

try:
    from numba import njit

    mandelbrot_numba = njit(mandelbrot_python)
    mandelbrot_numba(0.0, 0.0, 100)  # warm-up

    # Calculer une grille 100x100
    def grille_python():
        for x in range(-200, 200):
            for y in range(-200, 200):
                mandelbrot_python(x / 100, y / 100, 100)

    def grille_numba():
        for x in range(-200, 200):
            for y in range(-200, 200):
                mandelbrot_numba(x / 100, y / 100, 100)

    t_py = timeit.timeit(grille_python, number=1)
    t_nb = timeit.timeit(grille_numba, number=1)
    print(f"Python : {t_py:.3f}s")
    print(f"Numba  : {t_nb:.3f}s")
    print(f"Accélération : {t_py / t_nb:.0f}x")

except ImportError:
    print("numba non installé")
```

</details>

### Exercice 3 — Écrire un module Cython minimal *(moyen)*

Sans exécuter Cython (sauf si installé), écrire le contenu d'un fichier `distance.pyx` qui implémente le calcul de distance euclidienne entre deux points 3D, en utilisant les types C. Écrire aussi le `setup.py` de compilation.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Cython_numba_teasers", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
# distance.pyx
print("""
# distance.pyx
from libc.math cimport sqrt

def distance_3d(double x1, double y1, double z1,
                double x2, double y2, double z2) -> double:
    cdef double dx = x2 - x1
    cdef double dy = y2 - y1
    cdef double dz = z2 - z1
    return sqrt(dx * dx + dy * dy + dz * dz)
""")

# setup.py
print("""
# setup.py
from setuptools import setup
from Cython.Build import cythonize

setup(
    ext_modules=cythonize("distance.pyx"),
)

# Compilation : python setup.py build_ext --inplace
# ou : cythonize -i distance.pyx
""")
```

</details>

### Exercice 4 — Arbre de décision d'optimisation *(difficile)*

Vous avez une application web qui :
1. Reçoit des requêtes HTTP (I/O-bound) ;
2. Parse du JSON (CPU léger) ;
3. Fait un calcul numérique lourd sur les données (CPU-bound) ;
4. Écrit les résultats en base de données (I/O-bound).

Pour chaque étape, indiquez quelle technique d'optimisation est appropriée et pourquoi. Implémentez un prototype montrant le pipeline optimal.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="05_Cython_numba_teasers", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
# Analyse du pipeline :
#
# 1. Réception HTTP (I/O-bound)
#    → asyncio / aiohttp — la compilation n'aide pas
#
# 2. Parsing JSON (CPU léger)
#    → orjson (bibliothèque C) au lieu de json
#    → ou Python standard si pas le goulot
#
# 3. Calcul numérique lourd (CPU-bound)
#    → Numba @njit si boucles numériques
#    → NumPy si vectorisable
#    → multiprocessing.Pool pour paralléliser
#
# 4. Écriture base de données (I/O-bound)
#    → Batching (INSERT multiple)
#    → asyncio avec asyncpg / aiosqlite

import time
from concurrent.futures import ProcessPoolExecutor

def etape_io():
    time.sleep(0.01)  # simule I/O

def calcul_lourd(data):
    # En vrai : @njit ici
    return sum(x ** 2 for x in data)

def pipeline_sequentiel(requetes):
    resultats = []
    for req in requetes:
        etape_io()  # 1. réception
        data = list(range(req))  # 2. parsing
        result = calcul_lourd(data)  # 3. calcul
        etape_io()  # 4. écriture
        resultats.append(result)
    return resultats

# Le calcul peut être parallélisé
def pipeline_optimise(requetes):
    with ProcessPoolExecutor(max_workers=4) as pool:
        donnees = [list(range(r)) for r in requetes]
        resultats = list(pool.map(calcul_lourd, donnees))
    return resultats

requetes = [10_000] * 8
t1 = time.perf_counter()
pipeline_sequentiel(requetes)
t2 = time.perf_counter()
pipeline_optimise(requetes)
t3 = time.perf_counter()

print(f"Séquentiel : {t2-t1:.3f}s")
print(f"Optimisé   : {t3-t2:.3f}s")
```

</details>

---

## 9. Ressources

- [Numba — documentation officielle](https://numba.pydata.org/)
- [Cython — documentation officielle](https://cython.readthedocs.io/)
- [mypyc — documentation](https://mypyc.readthedocs.io/)
- [High Performance Python](https://www.oreilly.com/library/view/high-performance-python/9781492055013/) — Gorelick & Ozsvald
- [Python Speed Center](https://speed.python.org/)